In [ ]:
import matplotlib.pyplot as plt
from functools import reduce
from functools import partial
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
import h5py
import numpy as np
from timeit import default_timer
import operator
from datetime import date
from scipy.linalg import norm
from pathlib import Path

from PIL import Image


Mounted at /content/drive


In [ ]:
def determine_flow_parameters(u_x,u_y):
    # Determine number of sequences
    [n_sequences,sequence_length,x,y] = u_x.shape

    # Determine Re, U
    u_y_cont = np.reshape(u_y,(sequence_length*n_sequences,x,y))
    # Choose pixels to determine Re, U
    u_pxs_x,u_pxs_y = [84,89],[0,40]
    U = np.mean(u_x[:,:,u_pxs_x[0]:u_pxs_x[1],u_pxs_y[0]:u_pxs_y[1]])# flow speed, m/s
    U_std = np.std(u_x[:,:,u_pxs_x[0]:u_pxs_x[1],u_pxs_y[0]:u_pxs_y[1]])
    Re = U*D/kinematic_viscosity

    # St from FFT
    frame_rate = fps[trial]/5 # 80 fps frame rate using 1/5 frames, fps
    freq = np.fft.fftfreq(sequence_length*n_sequences,1/frame_rate)
    fft_pxs_x, fft_pxs_y = [48,52],[90,94]
    u_y_fft = np.mean(u_y_cont[:,fft_pxs_x[0]:fft_pxs_x[1],fft_pxs_y[0]:fft_pxs_y[1]],axis=(1,2))
    ft = np.abs(np.fft.fft(u_y_fft))[0:int(len(freq)/2)]
    maxes, idx = np.max(ft),np.argmax(ft)
    f_max = freq[idx]
    St = f_max*D/U
    print('   Reynolds number: '+str(round(Re,1)))
    print('   Strouhal number: '+str(round(St,2)))
    print('   Flow velocity: '+str(round(U,3))+'\u00B1'+str(round(U_std,3)))
    print('   Non-dimensional time: '+str(round(U/frame_rate/D,2)))
    print('   Shedding frequency: '+str(round(f_max,2)))
    print('   Shedding cycles per sequence: '+str(round(f_max*sequence_length/frame_rate,1)))
    print('   N sequences: '+str(n_sequences))

    return Re, St, U, n_sequences

def normalize(arr):
    # Normalize arrays to save stills, gifs
    arr_min,arr_max = np.min(arr), np.max(arr)
    return (arr-arr_min)/(arr_max-arr_min)

def save_stills(u,output_still_path,trial):
    plt.figure()
    plt.imshow(u,cmap='gray')
    plt.colorbar()
    plt.savefig(output_still_path)
    plt.close('all')

def create_gif(images, output_gif_path, duration=1/15):
    images = 255*normalize(images)
    imgs = [Image.fromarray(img).convert('RGB') for img in images]
    # Save as GIF
    imgs[0].save(
        output_gif_path,
        save_all=True,
        append_images=imgs[1:],
        duration=duration,
        loop=0 # 0 means infinite loop
    )



In [ ]:
flow_parameters = True
generate_stills = True
generate_gifs = True

trial_Re = ['240','650','910','1280','1890','2520']
fps = {'240':80,'650':200,'910':280,'1280':400,'1890':600,'2520':800}
inpath_prefix = '/content/drive/MyDrive/Dataset Paper/'
outpath_prefix = '/content/drive/MyDrive/Dataset Paper/outdata/'

# Constants
D = 9.525*10**-3 # cylinder diameter, m
kinematic_viscosity = 1e-6 # m^2/s


In [ ]:
if flow_parameters:
    # Empty arrays for storing data
    Re,St = np.zeros(len(trial_Re)),np.zeros(len(trial_Re))
    U,n_seq = np.zeros(len(trial_Re)),np.zeros(len(trial_Re))

if generate_stills:
    # Parameters relevant to saving stills
    still_sequences = {'240':4,'650':4,'910':0,'1280':8,'1890':5,'2520':8}
    still_timestep = {'240':0,'650':0,'910':0,'1280':0,'1890':0,'2520':0}
    # Code will generate a still of the flow field from the selected time step in the selected sequence

if generate_gifs:
    # Parameters relevant to saving stills
    gif_sequences = {'240':0,'650':0,'910':0,'1280':0,'1890':0,'2520':0}
    playback_speed = 1/15
    # Code will generate a gif of the flow field from the selected sequence



In [ ]:
for trial_idx,trial in enumerate(trial_Re):
    # Load data
    print('Loading data from Re = '+trial)
    fname = inpath_prefix+'Re_'+trial+'.h5'
    with h5py.File(fname, 'r') as f:
        u_x = f['u_x'][:]
        u_y = f['u_y'][:]
    # Structure: t, x, y, “index” (so each 100 x 89 x 133 x i represents one 100 step time sequence)

    if flow_parameters:
        print('Determining flow parameters')
        [Re[trial_idx],St[trial_idx],U[trial_idx],n_seq[trial_idx]] = determine_flow_parameters(u_x,u_y)

    if generate_stills:
        print('Generating stills')
        save_stills(u_x[still_sequences[trial],still_timestep[trial],:,:], \
            outpath_prefix+'still_Re'+trial+'_vu.png', trial)
        save_stills(u_y[still_sequences[trial],still_timestep[trial],:,:], \
            outpath_prefix+'still_Re'+trial+'_v.png', trial)

    if generate_gifs:
        print('Generating gifs')
        create_gif(u_x[gif_sequences[trial],:,:,:],outpath_prefix+'gif_Re'+trial+'_u.png')
        create_gif(u_y[gif_sequences[trial],:,:,:],outpath_prefix+'gif_Re'+trial+'_v.png')

if flow_parameters:
    # Create plot of St vs Re
    plt.figure()
    plt.plot(Re,St,'.-',color='black')
    plt.xlabel('Re')
    plt.ylabel('St')
    plt.grid('on')
    plt.savefig(outpath_prefix+'St_vs_Re.svg')
    plt.close('all')


Loading data from Re = 240
Determining flow parameters
   Reynolds number: 241.8
   Strouhal number: 0.16
   Flow velocity: 0.025±0.001
   Non-dimensional time: 0.17
   Shedding frequency: 0.42
   Shedding cycles per sequence: 2.6
   N sequences: 567
Generating stills
Generating gifs
Loading data from Re = 650
Determining flow parameters
   Reynolds number: 651.0
   Strouhal number: 0.16
   Flow velocity: 0.068±0.001
   Non-dimensional time: 0.18
   Shedding frequency: 1.13
   Shedding cycles per sequence: 2.8
   N sequences: 934
Generating stills
Generating gifs
Loading data from Re = 910
Determining flow parameters
   Reynolds number: 906.9
   Strouhal number: 0.15
   Flow velocity: 0.095±0.001
   Non-dimensional time: 0.18
   Shedding frequency: 1.54
   Shedding cycles per sequence: 2.8
   N sequences: 481
Generating stills
Generating gifs
Loading data from Re = 1280
Determining flow parameters
   Reynolds number: 1281.4
   Strouhal number: 0.17
   Flow velocity: 0.135±0.002
   Non-